In [ ]:
import pandas as pd
import os

# =====================================
# FILES
# =====================================
files = [
    "Farm 1.csv",
    "Farm 2.csv",
    "Farm 3.csv",
    "Farm 4.csv"
]

# =====================================
# SMW FUNCTION (STRICT ICAR)
# =====================================
def get_smw(date):

    m = date.month
    d = date.day

    if (m==1 and d<=7): return 1
    elif (m==1 and d<=14): return 2
    elif (m==1 and d<=21): return 3
    elif (m==1 and d<=28): return 4
    elif (m==1 and d>=29) or (m==2 and d<=4): return 5
    elif (m==2 and d<=11): return 6
    elif (m==2 and d<=18): return 7
    elif (m==2 and d<=25): return 8
    elif (m==2 and d>=26) or (m==3 and d<=4): return 9
    elif (m==3 and d<=11): return 10
    elif (m==3 and d<=18): return 11
    elif (m==3 and d<=25): return 12
    elif (m==3 and d>=26) or (m==4 and d<=1): return 13
    elif (m==4 and d<=8): return 14
    elif (m==4 and d<=15): return 15
    elif (m==4 and d<=22): return 16
    elif (m==4 and d<=29): return 17
    elif (m==4 and d>=30) or (m==5 and d<=6): return 18
    elif (m==5 and d<=13): return 19
    elif (m==5 and d<=20): return 20
    elif (m==5 and d<=27): return 21
    elif (m==5 and d>=28) or (m==6 and d<=3): return 22
    elif (m==6 and d<=10): return 23
    elif (m==6 and d<=17): return 24
    elif (m==6 and d<=24): return 25
    elif (m==6 and d>=25) or (m==7 and d<=1): return 26
    elif (m==7 and d<=8): return 27
    elif (m==7 and d<=15): return 28
    elif (m==7 and d<=22): return 29
    elif (m==7 and d<=29): return 30
    elif (m==7 and d>=30) or (m==8 and d<=5): return 31
    elif (m==8 and d<=12): return 32
    elif (m==8 and d<=19): return 33
    elif (m==8 and d<=26): return 34
    elif (m==8 and d>=27) or (m==9 and d<=2): return 35
    elif (m==9 and d<=9): return 36
    elif (m==9 and d<=16): return 37
    elif (m==9 and d<=23): return 38
    elif (m==9 and d<=30): return 39
    elif (m==10 and d<=7): return 40
    elif (m==10 and d<=14): return 41
    elif (m==10 and d<=21): return 42
    elif (m==10 and d<=28): return 43
    elif (m==10 and d>=29) or (m==11 and d<=4): return 44
    elif (m==11 and d<=11): return 45
    elif (m==11 and d<=18): return 46
    elif (m==11 and d<=25): return 47
    elif (m==11 and d>=26) or (m==12 and d<=2): return 48
    elif (m==12 and d<=9): return 49
    elif (m==12 and d<=16): return 50
    elif (m==12 and d<=23): return 51
    else: return 52


# =====================================
# PROCESS EACH FILE
# =====================================
for file in files:

    print("Processing:",file)

    # ---------------------------------
    # Read CSV (data starts from row 4)
    # ---------------------------------
    df = pd.read_csv(file, skiprows=3)

    # Clean column names
    df.columns = df.columns.str.lower().str.strip()

    # Find date column
    date_col = None
    for c in df.columns:
        if "time" in c or "date" in c:
            date_col = c
            break

    if date_col is None:
        print("Date column not found")
        continue

    df.rename(columns={date_col:"date"}, inplace=True)

    # Convert date
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")

    df = df.dropna(subset=["date"])

    # Create Year + SMW
    df["Year"] = df["date"].dt.year
    df["SMW"] = df["date"].apply(get_smw)

    # ---------------------------------
    # WEEKLY AGGREGATION
    # ---------------------------------
    numeric_cols = df.select_dtypes(include="number").columns.tolist()

    for c in ["Year","SMW"]:
        if c in numeric_cols:
            numeric_cols.remove(c)

    agg = {col:"mean" for col in numeric_cols}

    weekly = df.groupby(["Year","SMW"]).agg(agg).reset_index()

    # ---------------------------------
    # Ensure continuous SMW weeks
    # ---------------------------------
    years = weekly["Year"].unique()

    full_index = pd.MultiIndex.from_product(
        [years, range(1,53)],
        names=["Year","SMW"]
    )

    weekly = weekly.set_index(["Year","SMW"]).reindex(full_index).reset_index()

    # ---------------------------------
    # Save
    # ---------------------------------
    out = file.replace(".csv","_SMW_weekly.csv")

    weekly.to_csv(out,index=False)

    print("Saved:",out)

print("All farms processed successfully.")

Processing: Farm 1.csv
Saved: Farm 1_SMW_weekly.csv
Processing: Farm 2.csv
Saved: Farm 2_SMW_weekly.csv
Processing: Farm 3.csv
Saved: Farm 3_SMW_weekly.csv
Processing: Farm 4.csv
Saved: Farm 4_SMW_weekly.csv
All farms processed successfully.
